# Airline On-Time — cleaning & EDA

Quick pass over the Jan 2024 BTS sample before the Power BI model.

Goal: confirm null patterns, cancellation share, and that on-time % lines up with the dashboard (~75.8%).


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data')
OUT = Path('outputs')
OUT.mkdir(parents=True, exist_ok=True)

flights = pd.read_csv(DATA / 'fact_flights.csv')
print('rows', len(flights), 'cols', list(flights.columns)[:12])
flights.head()


In [ ]:
op = flights[(flights['Cancelled'] == 0) & (flights['Diverted'] == 0)]
print('flights', len(flights))
print('on_time_pct', round(op['IsOnTimeArr'].mean() * 100, 1))
print('avg_arr_delay', round(flights['ArrDelay'].clip(lower=0).mean(), 1))
print('cancel_pct', round(flights['Cancelled'].mean() * 100, 1))


In [ ]:
# Delay cause mix (where columns exist)
cause_cols = [c for c in ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay'] if c in flights.columns]
if cause_cols:
    causes = flights[cause_cols].fillna(0).sum().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(causes.index, causes.values)
    ax.set_title('Delay minutes by cause')
    fig.tight_layout()
    fig.savefig(OUT / 'delay_cause_mix.png', dpi=120)
    plt.show()


In [ ]:
if 'FlightDate' in flights.columns:
    daily = op.groupby('FlightDate')['IsOnTimeArr'].mean().reset_index()
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(pd.to_datetime(daily['FlightDate']), daily['IsOnTimeArr'] * 100)
    ax.set_title('Daily on-time arrival %')
    ax.set_ylabel('%')
    fig.tight_layout()
    fig.savefig(OUT / 'daily_ontime.png', dpi=120)
    plt.show()
elif 'DayofMonth' in flights.columns:
    daily = op.groupby('DayofMonth')['IsOnTimeArr'].mean()
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(daily.index, daily.values * 100, marker='o')
    ax.set_title('On-time % by day of month')
    fig.tight_layout()
    fig.savefig(OUT / 'daily_ontime.png', dpi=120)
    plt.show()
